In [1]:
# # Molecular Docking: Peptide Ligands vs VEGF Receptor
# 
# This notebook performs automated molecular docking of peptide ligands against the VEGF receptor
# using AutoDock Vina, analyzes binding poses, and generates comprehensive reports.

# ## 1. Import Required Libraries and Setup

import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Image
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Molecular visualization and analysis
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors
    from rdkit.Chem import Draw
    print("✓ RDKit imported successfully")
except ImportError:
    print("⚠ RDKit not available - some features will be limited")

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

# Create directory structure for docking
base_dir = Path("../data/processed/DockingPrep")
ligands_dir = base_dir / "ligands"
receptor_dir = base_dir / "receptor"
results_dir = base_dir / "docking_results"
analysis_dir = base_dir / "analysis"

# Create all directories
for directory in [receptor_dir, results_dir, analysis_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"✓ Directory structure created:")
print(f"   Ligands: {ligands_dir}")
print(f"   Receptor: {receptor_dir}")
print(f"   Results: {results_dir}")
print(f"   Analysis: {analysis_dir}")

✓ RDKit imported successfully
✓ Directory structure created:
   Ligands: DockingPrep\ligands
   Receptor: DockingPrep\receptor
   Results: DockingPrep\docking_results
   Analysis: DockingPrep\analysis


In [3]:
# ## 2. Configuration and File Validation

# Configuration
RECEPTOR_FILE = "1VPF_receptor.pdbqt"
RECEPTOR_PATH = receptor_dir / RECEPTOR_FILE

# AutoDock Vina parameters
VINA_CONFIG = {
    'exhaustiveness': 8,
    'num_modes': 9,
    'energy_range': 3.0,
    'cpu': 4  # Adjust based on your system
}

# Binding site configuration (will be auto-detected or manually set)
BINDING_SITE = {
    'center_x': 0.0,
    'center_y': 0.0, 
    'center_z': 0.0,
    'size_x': 20.0,
    'size_y': 20.0,
    'size_z': 20.0
}

print(f"AutoDock Vina Configuration:")
for key, value in VINA_CONFIG.items():
    print(f"   {key}: {value}")

# ## 3. Utility Functions for Docking

class MolecularDocking:
    """
    Comprehensive molecular docking class using AutoDock Vina.
    """
    
    def __init__(self, receptor_path, ligands_dir, results_dir):
        self.receptor_path = Path(receptor_path)
        self.ligands_dir = Path(ligands_dir)
        self.results_dir = Path(results_dir)
        self.results_dir.mkdir(exist_ok=True)
        
        # Verify AutoDock Vina installation
        self.vina_available = self.check_vina_installation()
        
    def check_vina_installation(self):
        """Check if AutoDock Vina is available."""
        try:
            result = subprocess.run(['vina', '--help'], 
                                  capture_output=True, text=True, timeout=10)
            if result.returncode == 0:
                print("✓ AutoDock Vina found and working")
                return True
            else:
                print("✗ AutoDock Vina not working properly")
                return False
        except (subprocess.TimeoutExpired, FileNotFoundError):
            print("✗ AutoDock Vina not found in PATH")
            print("   Please install AutoDock Vina: conda install -c conda-forge autodock-vina")
            return False
    
    def convert_mol2_to_pdbqt(self, mol2_file):
        """Convert MOL2 file to PDBQT format using OpenBabel."""
        mol2_path = Path(mol2_file)
        pdbqt_path = mol2_path.parent / f"{mol2_path.stem}.pdbqt"
        
        try:
            # Use OpenBabel for conversion
            cmd = ['obabel', '-imol2', str(mol2_path), '-opdbqt', '-O', str(pdbqt_path)]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
            
            if result.returncode == 0 and pdbqt_path.exists():
                print(f"   ✓ Converted: {mol2_path.name} → {pdbqt_path.name}")
                return pdbqt_path
            else:
                print(f"   ✗ Conversion failed for {mol2_path.name}")
                print(f"     Error: {result.stderr}")
                return None
                
        except subprocess.TimeoutExpired:
            print(f"   ✗ Conversion timeout for {mol2_path.name}")
            return None
        except FileNotFoundError:
            print("   ✗ OpenBabel not found. Please install: conda install -c conda-forge openbabel")
            return None
    
    def detect_binding_site(self, method='center_of_mass'):
        """
        Detect binding site from receptor structure.
        Methods: 'center_of_mass', 'cavity_detection', 'manual'
        """
        if not self.receptor_path.exists():
            print(f"✗ Receptor file not found: {self.receptor_path}")
            return None
        
        try:
            if method == 'center_of_mass':
                # Simple center of mass calculation from PDBQT
                coords = []
                with open(self.receptor_path, 'r') as f:
                    for line in f:
                        if line.startswith('ATOM') or line.startswith('HETATM'):
                            try:
                                x = float(line[30:38].strip())
                                y = float(line[38:46].strip())
                                z = float(line[46:54].strip())
                                coords.append([x, y, z])
                            except:
                                continue
                
                if coords:
                    coords = np.array(coords)
                    center = np.mean(coords, axis=0)
                    
                    # Calculate reasonable box size based on protein dimensions
                    min_coords = np.min(coords, axis=0)
                    max_coords = np.max(coords, axis=0)
                    dimensions = max_coords - min_coords
                    
                    # Use 50% of protein dimensions with minimum of 20Å
                    box_size = np.maximum(dimensions * 0.5, 20.0)
                    
                    binding_site = {
                        'center_x': float(center[0]),
                        'center_y': float(center[1]),
                        'center_z': float(center[2]),
                        'size_x': float(box_size[0]),
                        'size_y': float(box_size[1]),
                        'size_z': float(box_size[2])
                    }
                    
                    print(f"✓ Binding site detected using {method}:")
                    print(f"   Center: ({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f})")
                    print(f"   Box size: ({box_size[0]:.2f}, {box_size[1]:.2f}, {box_size[2]:.2f})")
                    
                    return binding_site
                else:
                    print("✗ No coordinates found in receptor file")
                    return None
                    
        except Exception as e:
            print(f"✗ Binding site detection failed: {e}")
            return None
    
    def create_vina_config(self, ligand_pdbqt, output_pdbqt, binding_site=None):
        """Create AutoDock Vina configuration file."""
        if binding_site is None:
            binding_site = BINDING_SITE
        
        config_file = self.results_dir / f"{Path(ligand_pdbqt).stem}_config.txt"
        
        config_content = f"""# AutoDock Vina configuration file
receptor = {self.receptor_path.absolute()}
ligand = {ligand_pdbqt}
out = {output_pdbqt}

center_x = {binding_site['center_x']:.3f}
center_y = {binding_site['center_y']:.3f}
center_z = {binding_site['center_z']:.3f}

size_x = {binding_site['size_x']:.1f}
size_y = {binding_site['size_y']:.1f}
size_z = {binding_site['size_z']:.1f}

exhaustiveness = {VINA_CONFIG['exhaustiveness']}
num_modes = {VINA_CONFIG['num_modes']}
energy_range = {VINA_CONFIG['energy_range']}
cpu = {VINA_CONFIG['cpu']}
"""
        
        with open(config_file, 'w') as f:
            f.write(config_content)
        
        return config_file
    
    def run_vina_docking(self, ligand_pdbqt, binding_site=None):
        """Run AutoDock Vina docking for a single ligand."""
        ligand_name = Path(ligand_pdbqt).stem
        output_pdbqt = self.results_dir / f"{ligand_name}_docked.pdbqt"
        log_file = self.results_dir / f"{ligand_name}_docking.log"
        
        # Create configuration file
        config_file = self.create_vina_config(ligand_pdbqt, output_pdbqt, binding_site)
        
        try:
            # Run AutoDock Vina
            cmd = ['vina', '--config', str(config_file)]
            
            with open(log_file, 'w') as log:
                result = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, 
                                      timeout=300, text=True)
            
            if result.returncode == 0 and output_pdbqt.exists():
                print(f"   ✓ Docking completed: {ligand_name}")
                
                # Parse results
                scores = self.parse_vina_output(log_file)
                return {
                    'ligand': ligand_name,
                    'output_file': output_pdbqt,
                    'log_file': log_file,
                    'scores': scores,
                    'status': 'success'
                }
            else:
                print(f"   ✗ Docking failed: {ligand_name}")
                return {
                    'ligand': ligand_name,
                    'status': 'failed',
                    'error': 'Vina execution failed'
                }
                
        except subprocess.TimeoutExpired:
            print(f"   ✗ Docking timeout: {ligand_name}")
            return {
                'ligand': ligand_name,
                'status': 'timeout',
                'error': 'Docking timeout (>5 min)'
            }
        except Exception as e:
            print(f"   ✗ Docking error: {ligand_name} - {e}")
            return {
                'ligand': ligand_name,
                'status': 'error',
                'error': str(e)
            }
    
    def parse_vina_output(self, log_file):
        """Parse AutoDock Vina output log to extract binding scores."""
        scores = []
        
        try:
            with open(log_file, 'r') as f:
                lines = f.readlines()
            
            # Find the results section
            in_results = False
            for line in lines:
                if 'mode |   affinity | dist from best mode' in line:
                    in_results = True
                    continue
                elif in_results and line.strip().startswith('-----'):
                    break
                elif in_results and line.strip():
                    parts = line.strip().split()
                    if len(parts) >= 3 and parts[0].isdigit():
                        try:
                            mode = int(parts[0])
                            affinity = float(parts[1])
                            rmsd_lb = float(parts[2]) if len(parts) > 2 else 0.0
                            rmsd_ub = float(parts[3]) if len(parts) > 3 else 0.0
                            
                            scores.append({
                                'mode': mode,
                                'affinity': affinity,
                                'rmsd_lb': rmsd_lb,
                                'rmsd_ub': rmsd_ub
                            })
                        except ValueError:
                            continue
            
            return scores
            
        except Exception as e:
            print(f"   Warning: Could not parse log file {log_file}: {e}")
            return []
    
    def dock_all_ligands(self, binding_site=None):
        """Dock all available ligands against the receptor."""
        # Find all MOL2 files
        mol2_files = list(self.ligands_dir.glob("*.mol2"))
        
        if not mol2_files:
            print("✗ No MOL2 files found in ligands directory")
            return []
        
        print(f"Found {len(mol2_files)} ligand files to dock")
        
        if not self.vina_available:
            print("✗ Cannot proceed without AutoDock Vina")
            return []
        
        if not self.receptor_path.exists():
            print(f"✗ Receptor file not found: {self.receptor_path}")
            return []
        
        # Detect binding site if not provided
        if binding_site is None:
            binding_site = self.detect_binding_site()
            if binding_site is None:
                print("✗ Could not detect binding site, using default")
                binding_site = BINDING_SITE
        
        # Convert MOL2 to PDBQT and dock
        results = []
        
        for i, mol2_file in enumerate(mol2_files, 1):
            print(f"\n[{i}/{len(mol2_files)}] Processing {mol2_file.name}")
            
            # Convert to PDBQT
            pdbqt_file = self.convert_mol2_to_pdbqt(mol2_file)
            
            if pdbqt_file:
                # Run docking
                result = self.run_vina_docking(pdbqt_file, binding_site)
                results.append(result)
            else:
                results.append({
                    'ligand': mol2_file.stem,
                    'status': 'conversion_failed',
                    'error': 'MOL2 to PDBQT conversion failed'
                })
        
        return results
# Initialize docking class
docking = MolecularDocking(RECEPTOR_PATH, ligands_dir, results_dir)

AutoDock Vina Configuration:
   exhaustiveness: 8
   num_modes: 9
   energy_range: 3.0
   cpu: 4
✓ AutoDock Vina found and working


In [4]:
# ## 4. File Validation and Preparation

print(f"{'='*60}")
print("FILE VALIDATION AND PREPARATION")
print(f"{'='*60}")

# Check receptor file
if RECEPTOR_PATH.exists():
    file_size = RECEPTOR_PATH.stat().st_size
    print(f"✓ Receptor file found: {RECEPTOR_FILE} ({file_size} bytes)")
    
    # Basic validation of PDBQT format
    with open(RECEPTOR_PATH, 'r') as f:
        content = f.read()
        if 'ATOM' in content and 'END' in content:
            print("✓ Receptor file appears to be valid PDBQT format")
        else:
            print("⚠ Warning: Receptor file may not be properly formatted")
else:
    print(f"✗ Receptor file not found: {RECEPTOR_PATH}")
    print(f"   Please place your {RECEPTOR_FILE} in: {receptor_dir}")
    print("   Creating placeholder file...")
    
    # Create a placeholder message
    RECEPTOR_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(RECEPTOR_PATH, 'w') as f:
        f.write(f"""# Placeholder for VEGF receptor file
# Please replace this file with your actual {RECEPTOR_FILE}
# 
# The file should be in PDBQT format with:
# - ATOM records for protein coordinates
# - Proper AutoDock atom types
# - Partial charges
# - END record at the end
""")
    print(f"   Placeholder created. Please replace with actual receptor file.")

# Check ligand files
mol2_files = list(ligands_dir.glob("*.mol2"))
print(f"\nLigand files found: {len(mol2_files)}")

if mol2_files:
    ligand_info = []
    for mol2_file in sorted(mol2_files):
        file_size = mol2_file.stat().st_size
        ligand_info.append({
            'Filename': mol2_file.name,
            'Size (bytes)': file_size,
            'Path': str(mol2_file)
        })
        print(f"  ✓ {mol2_file.name} ({file_size} bytes)")
    
    # Create ligand summary
    df_ligands = pd.DataFrame(ligand_info)
    print(f"\nLigand Summary:")
    display(df_ligands)
else:
    print("✗ No MOL2 ligand files found")
    print(f"   Please ensure your peptide MOL2 files are in: {ligands_dir}")
    print("   Expected files: peptide_1.mol2, peptide_2.mol2, etc.")


FILE VALIDATION AND PREPARATION
✓ Receptor file found: 1VPF_receptor.pdbqt (482742 bytes)
⚠ Warning: Receptor file may not be properly formatted

Ligand files found: 6
  ✓ peptide_1.mol2 (2295 bytes)
  ✓ peptide_2.mol2 (2848 bytes)
  ✓ peptide_3.mol2 (2848 bytes)
  ✓ peptide_4.mol2 (2848 bytes)
  ✓ peptide_5.mol2 (2420 bytes)
  ✓ peptide_6.mol2 (2420 bytes)

Ligand Summary:


,Filename,Size (bytes),Path
0,peptide_1.mol2,2295,DockingPrep\ligands\peptide_1.mol2
1,peptide_2.mol2,2848,DockingPrep\ligands\peptide_2.mol2
2,peptide_3.mol2,2848,DockingPrep\ligands\peptide_3.mol2
3,peptide_4.mol2,2848,DockingPrep\ligands\peptide_4.mol2
4,peptide_5.mol2,2420,DockingPrep\ligands\peptide_5.mol2
5,peptide_6.mol2,2420,DockingPrep\ligands\peptide_6.mol2


In [5]:
# ## 5. Binding Site Analysis and Detection

print(f"\n{'='*60}")
print("BINDING SITE ANALYSIS")
print(f"{'='*60}")

if RECEPTOR_PATH.exists() and RECEPTOR_PATH.stat().st_size > 100:
    # Detect binding site
    detected_site = docking.detect_binding_site()
    
    if detected_site:
        # Update global binding site
        BINDING_SITE.update(detected_site)
        
        print(f"\n✓ Using detected binding site:")
        print(f"   Center: ({BINDING_SITE['center_x']:.2f}, {BINDING_SITE['center_y']:.2f}, {BINDING_SITE['center_z']:.2f})")
        print(f"   Box size: ({BINDING_SITE['size_x']:.1f} × {BINDING_SITE['size_y']:.1f} × {BINDING_SITE['size_z']:.1f}) Ų")
        
        # Visualize binding site (text representation)
        print(f"\nBinding Site Visualization:")
        print(f"```")
        print(f"     Z-axis")
        print(f"       |")
        print(f"       |  ┌─────────┐")
        print(f"       | /         /|  Box size: {BINDING_SITE['size_x']:.1f} × {BINDING_SITE['size_y']:.1f} × {BINDING_SITE['size_z']:.1f}")
        print(f"       |/         / |  Center: ({BINDING_SITE['center_x']:.1f}, {BINDING_SITE['center_y']:.1f}, {BINDING_SITE['center_z']:.1f})")
        print(f"       ┌─────────┐  |")
        print(f"       |    ●    |  |  ● = binding site center")
        print(f"       |         |  /")
        print(f"       |         | /")
        print(f"       └─────────┘/")
        print(f"       Y-axis ────────→ X-axis")
        print(f"```")
        
    else:
        print("⚠ Using default binding site coordinates")
        print(f"   You may need to manually adjust the binding site in the configuration")
else:
    print("⚠ Cannot analyze binding site - receptor file not available")

# ## 6. Molecular Docking Execution

print(f"\n{'='*60}")
print("MOLECULAR DOCKING EXECUTION")
print(f"{'='*60}")

# Check if we can proceed with docking
can_dock = (
    RECEPTOR_PATH.exists() and 
    RECEPTOR_PATH.stat().st_size > 100 and 
    len(mol2_files) > 0 and 
    docking.vina_available
)

if can_dock:
    print("✓ All requirements met. Starting molecular docking...")
    
    # Run docking for all ligands
    docking_results = docking.dock_all_ligands(BINDING_SITE)
    
    print(f"\n{'='*60}")
    print("DOCKING RESULTS SUMMARY")
    print(f"{'='*60}")
    
    # Analyze results
    successful_dockings = [r for r in docking_results if r['status'] == 'success']
    failed_dockings = [r for r in docking_results if r['status'] != 'success']
    
    print(f"Total ligands processed: {len(docking_results)}")
    print(f"Successful dockings: {len(successful_dockings)}")
    print(f"Failed dockings: {len(failed_dockings)}")
    
    if failed_dockings:
        print(f"\nFailed docking details:")
        for result in failed_dockings:
            print(f"  ✗ {result['ligand']}: {result.get('error', 'Unknown error')}")
    
else:
    print("✗ Cannot proceed with docking. Missing requirements:")
    if not RECEPTOR_PATH.exists():
        print(f"   - Receptor file: {RECEPTOR_FILE}")
    if len(mol2_files) == 0:
        print("   - Ligand MOL2 files")
    if not docking.vina_available:
        print("   - AutoDock Vina installation")
    
    print("\nCreating mock results for demonstration...")
    
    # Create mock results for testing
    docking_results = []
    mock_peptides = ['peptide_1', 'peptide_2', 'peptide_3', 'peptide_4', 'peptide_5', 'peptide_6']
    
    for i, peptide in enumerate(mock_peptides):
        # Simulate binding scores (more negative = better binding)
        mock_scores = []
        for mode in range(1, 4):  # 3 modes
            affinity = np.random.uniform(-8.5, -5.2)  # Typical range for peptides
            rmsd_lb = np.random.uniform(0.0, 2.5)
            rmsd_ub = np.random.uniform(rmsd_lb, rmsd_lb + 3.0)
            
            mock_scores.append({
                'mode': mode,
                'affinity': round(affinity, 1),
                'rmsd_lb': round(rmsd_lb, 1),
                'rmsd_ub': round(rmsd_ub, 1)
            })
        
        docking_results.append({
            'ligand': peptide,
            'status': 'success',
            'scores': mock_scores
        })
    
    successful_dockings = docking_results
    print(f"✓ Created mock results for {len(docking_results)} peptides")


BINDING SITE ANALYSIS
✓ Binding site detected using center_of_mass:
   Center: (5.95, 2.62, 28.66)
   Box size: (44.14, 20.00, 42.22)

✓ Using detected binding site:
   Center: (5.95, 2.62, 28.66)
   Box size: (44.1 × 20.0 × 42.2) Ų

Binding Site Visualization:
```
     Z-axis
       |
       |  ┌─────────┐
       | /         /|  Box size: 44.1 × 20.0 × 42.2
       |/         / |  Center: (5.9, 2.6, 28.7)
       ┌─────────┐  |
       |    ●    |  |  ● = binding site center
       |         |  /
       |         | /
       └─────────┘/
       Y-axis ────────→ X-axis
```

MOLECULAR DOCKING EXECUTION
✓ All requirements met. Starting molecular docking...
Found 6 ligand files to dock

[1/6] Processing peptide_1.mol2
   ✓ Converted: peptide_1.mol2 → peptide_1.pdbqt
   ✓ Docking completed: peptide_1

[2/6] Processing peptide_2.mol2
   ✓ Converted: peptide_2.mol2 → peptide_2.pdbqt
   ✓ Docking completed: peptide_2

[3/6] Processing peptide_3.mol2
   ✓ Converted: peptide_3.mol2 → peptide_3.pdb

In [6]:
# ## 7. Results Analysis and Visualization

if docking_results:
    print(f"\n{'='*60}")
    print("DOCKING RESULTS ANALYSIS")
    print(f"{'='*60}")
    
    # Create comprehensive results DataFrame
    analysis_data = []
    
    for result in successful_dockings:
        ligand_name = result['ligand']
        scores = result.get('scores', [])
        
        if scores:
            # Best binding pose (lowest affinity)
            best_pose = min(scores, key=lambda x: x['affinity'])
            
            analysis_data.append({
                'Ligand': ligand_name,
                'Best_Affinity': best_pose['affinity'],
                'Best_Mode': best_pose['mode'],
                'RMSD_lb': best_pose['rmsd_lb'],
                'RMSD_ub': best_pose['rmsd_ub'],
                'Num_Poses': len(scores),
                'Affinity_Range': max(scores, key=lambda x: x['affinity'])['affinity'] - best_pose['affinity'],
                'Status': 'Success'
            })
        else:
            analysis_data.append({
                'Ligand': ligand_name,
                'Best_Affinity': np.nan,
                'Best_Mode': np.nan,
                'RMSD_lb': np.nan,
                'RMSD_ub': np.nan,
                'Num_Poses': 0,
                'Affinity_Range': np.nan,
                'Status': 'No_Scores'
            })
    
    # Add failed dockings
    for result in failed_dockings:
        analysis_data.append({
            'Ligand': result['ligand'],
            'Best_Affinity': np.nan,
            'Best_Mode': np.nan,
            'RMSD_lb': np.nan,
            'RMSD_ub': np.nan,
            'Num_Poses': 0,
            'Affinity_Range': np.nan,
            'Status': f"Failed: {result.get('error', 'Unknown')}"
        })
    
    # Create results DataFrame
    df_results = pd.DataFrame(analysis_data)
    
    print("Docking Results Summary:")
    display(df_results)
    
    # Statistical analysis of successful dockings
    successful_df = df_results[df_results['Status'] == 'Success'].copy()
    
    if len(successful_df) > 0:
        print(f"\nBinding Affinity Statistics (n={len(successful_df)}):")
        affinity_stats = successful_df['Best_Affinity'].describe()
        display(affinity_stats.round(2))
        
        # Identify top binders
        top_binders = successful_df.nsmallest(3, 'Best_Affinity')
        print(f"\n🏆 Top 3 Binding Peptides:")
        for idx, row in top_binders.iterrows():
            print(f"   {row['Ligand']}: {row['Best_Affinity']:.1f} kcal/mol")
        
        # Binding classification
        strong_binders = successful_df[successful_df['Best_Affinity'] < -7.0]
        moderate_binders = successful_df[(successful_df['Best_Affinity'] >= -7.0) & 
                                       (successful_df['Best_Affinity'] < -5.5)]
        weak_binders = successful_df[successful_df['Best_Affinity'] >= -5.5]
        
        print(f"\nBinding Classification:")
        print(f"   Strong binders (<-7.0 kcal/mol): {len(strong_binders)}")
        print(f"   Moderate binders (-7.0 to -5.5 kcal/mol): {len(moderate_binders)}")
        print(f"   Weak binders (>-5.5 kcal/mol): {len(weak_binders)}")
        
        # Visualizations
        if len(successful_df) > 1:
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            fig.suptitle('Molecular Docking Results Analysis', fontsize=16)
            
            # Binding affinity distribution
            axes[0,0].hist(successful_df['Best_Affinity'], bins=10, alpha=0.7, 
                          color='skyblue', edgecolor='black')
            axes[0,0].axvline(successful_df['Best_Affinity'].mean(), color='red', 
                             linestyle='--', label=f'Mean: {successful_df["Best_Affinity"].mean():.1f}')
            axes[0,0].set_xlabel('Binding Affinity (kcal/mol)')
            axes[0,0].set_ylabel('Count')
            axes[0,0].set_title('Binding Affinity Distribution')
            axes[0,0].legend()
            
            # Binding affinity by ligand
            axes[0,1].bar(range(len(successful_df)), successful_df['Best_Affinity'], 
                         alpha=0.7, color='lightgreen', edgecolor='black')
            axes[0,1].set_xlabel('Ligand Index')
            axes[0,1].set_ylabel('Binding Affinity (kcal/mol)')
            axes[0,1].set_title('Binding Affinity by Ligand')
            axes[0,1].set_xticks(range(len(successful_df)))
            axes[0,1].set_xticklabels([l.replace('peptide_', 'P') for l in successful_df['Ligand']], 
                                     rotation=45)
            
            # RMSD analysis
            axes[1,0].scatter(successful_df['Best_Affinity'], successful_df['RMSD_lb'], 
                             alpha=0.7, s=60, color='orange')
            axes[1,0].set_xlabel('Binding Affinity (kcal/mol)')
            axes[1,0].set_ylabel('RMSD Lower Bound (Ų)')
            axes[1,0].set_title('Affinity vs RMSD')
            
            # Pose diversity
            axes[1,1].bar(range(len(successful_df)), successful_df['Num_Poses'], 
                         alpha=0.7, color='purple', edgecolor='black')
            axes[1,1].set_xlabel('Ligand Index')
            axes[1,1].set_ylabel('Number of Poses')
            axes[1,1].set_title('Binding Pose Diversity')
            axes[1,1].set_xticks(range(len(successful_df)))
            axes[1,1].set_xticklabels([l.replace('peptide_', 'P') for l in successful_df['Ligand']], 
                                     rotation=45)
            
            plt.tight_layout()
            plt.show()
        
        # Detailed pose analysis
        print(f"\n{'='*40}")
        print("DETAILED POSE ANALYSIS")
        print(f"{'='*40}")
        
        for result in successful_dockings[:3]:  # Show top 3
            ligand_name = result['ligand']
            scores = result.get('scores', [])
            
            if scores:
                print(f"\n{ligand_name}:")
                print(f"   Mode | Affinity | RMSD_lb | RMSD_ub")
                print(f"   -----|----------|---------|--------")
                for score in scores:
                    print(f"   {score['mode']:4d} | {score['affinity']:8.1f} | {score['rmsd_lb']:7.1f} | {score['rmsd_ub']:7.1f}")



DOCKING RESULTS ANALYSIS
Docking Results Summary:


,Ligand,Best_Affinity,Best_Mode,RMSD_lb,RMSD_ub,Num_Poses,Affinity_Range,Status
0,peptide_1,NaN,NaN,NaN,NaN,0,NaN,No_Scores
1,peptide_2,NaN,NaN,NaN,NaN,0,NaN,No_Scores
2,peptide_3,NaN,NaN,NaN,NaN,0,NaN,No_Scores
3,peptide_4,NaN,NaN,NaN,NaN,0,NaN,No_Scores
4,peptide_5,NaN,NaN,NaN,NaN,0,NaN,No_Scores
5,peptide_6,NaN,NaN,NaN,NaN,0,NaN,No_Scores


In [7]:
# ## 8. Structure-Activity Relationship (SAR) Analysis

if len(successful_df) > 0:
    print(f"\n{'='*60}")
    print("STRUCTURE-ACTIVITY RELATIONSHIP (SAR) ANALYSIS")
    print(f"{'='*60}")
    
    # Load peptide sequences if available
    peptide_sequences = {}
    csv_file = ligands_dir.parent / "peptides.csv"
    
    if csv_file.exists():
        try:
            df_peptides = pd.read_csv(csv_file)
            if 'Sequence' in df_peptides.columns:
                for idx, row in df_peptides.iterrows():
                    peptide_id = f"peptide_{idx + 1}"
                    peptide_sequences[peptide_id] = row['Sequence']
                print(f"✓ Loaded {len(peptide_sequences)} peptide sequences")
        except Exception as e:
            print(f"⚠ Could not load peptide sequences: {e}")
    
    # Add sequence information to results
    if peptide_sequences:
        successful_df['Sequence'] = successful_df['Ligand'].map(peptide_sequences)
        successful_df['Length'] = successful_df['Sequence'].str.len()
        
        # Amino acid composition analysis
        aa_composition = {}
        for seq in successful_df['Sequence'].dropna():
            for aa in seq:
                aa_composition[aa] = aa_composition.get(aa, 0) + 1
        
        print(f"\nAmino Acid Frequency in Dataset:")
        sorted_aa = sorted(aa_composition.items(), key=lambda x: x[1], reverse=True)
        for aa, count in sorted_aa[:10]:  # Top 10
            print(f"   {aa}: {count} occurrences")
        
        # Correlation analysis
        if len(successful_df) > 3:
            # Length vs binding affinity
            length_corr = successful_df['Length'].corr(successful_df['Best_Affinity'])
            print(f"\nSequence length vs binding affinity correlation: {length_corr:.3f}")
            
            # Hydrophobic residue analysis
            hydrophobic_aas = set('AILMFWYV')
            successful_df['Hydrophobic_Content'] = successful_df['Sequence'].apply(
                lambda seq: sum(1 for aa in seq if aa in hydrophobic_aas) / len(seq) if pd.notna(seq) else 0
            )
            
            hydrophobic_corr = successful_df['Hydrophobic_Content'].corr(successful_df['Best_Affinity'])
            print(f"Hydrophobic content vs binding affinity correlation: {hydrophobic_corr:.3f}")
            
            # Charged residue analysis
            positive_aas = set('RK')
            negative_aas = set('DE')
            successful_df['Positive_Content'] = successful_df['Sequence'].apply(
                lambda seq: sum(1 for aa in seq if aa in positive_aas) / len(seq) if pd.notna(seq) else 0
            )
            successful_df['Negative_Content'] = successful_df['Sequence'].apply(
                lambda seq: sum(1 for aa in seq if aa in negative_aas) / len(seq) if pd.notna(seq) else 0
            )
            
            # SAR visualization
            if len(successful_df) > 1:
                fig, axes = plt.subplots(2, 2, figsize=(14, 10))
                fig.suptitle('Structure-Activity Relationship Analysis', fontsize=16)
                
                # Length vs affinity
                axes[0,0].scatter(successful_df['Length'], successful_df['Best_Affinity'], 
                                alpha=0.7, s=80, color='blue')
                axes[0,0].set_xlabel('Peptide Length (AA)')
                axes[0,0].set_ylabel('Binding Affinity (kcal/mol)')
                axes[0,0].set_title(f'Length vs Affinity (r={length_corr:.3f})')
                
                # Add trend line
                if not successful_df['Length'].isna().all():
                    z = np.polyfit(successful_df['Length'].dropna(), 
                                 successful_df['Best_Affinity'][successful_df['Length'].notna()], 1)
                    p = np.poly1d(z)
                    axes[0,0].plot(successful_df['Length'], p(successful_df['Length']), "r--", alpha=0.8)
                
                # Hydrophobic content vs affinity
                axes[0,1].scatter(successful_df['Hydrophobic_Content'], successful_df['Best_Affinity'], 
                                alpha=0.7, s=80, color='green')
                axes[0,1].set_xlabel('Hydrophobic Content (fraction)')
                axes[0,1].set_ylabel('Binding Affinity (kcal/mol)')
                axes[0,1].set_title(f'Hydrophobicity vs Affinity (r={hydrophobic_corr:.3f})')
                
                # Positive charge content
                axes[1,0].scatter(successful_df['Positive_Content'], successful_df['Best_Affinity'], 
                                alpha=0.7, s=80, color='red')
                axes[1,0].set_xlabel('Positive Charge Content (fraction)')
                axes[1,0].set_ylabel('Binding Affinity (kcal/mol)')
                axes[1,0].set_title('Positive Charge vs Affinity')
                
                # Amino acid composition heatmap
                if len(sorted_aa) > 5:
                    aa_matrix = []
                    aa_labels = []
                    affinity_values = []
                    
                    for _, row in successful_df.iterrows():
                        if pd.notna(row['Sequence']):
                            aa_counts = {aa: 0 for aa, _ in sorted_aa[:8]}  # Top 8 AAs
                            for aa in row['Sequence']:
                                if aa in aa_counts:
                                    aa_counts[aa] += 1
                            
                            # Normalize by sequence length
                            total_len = len(row['Sequence'])
                            aa_fractions = [aa_counts[aa] / total_len for aa, _ in sorted_aa[:8]]
                            aa_matrix.append(aa_fractions)
                            aa_labels.append(row['Ligand'])
                            affinity_values.append(row['Best_Affinity'])
                    
                    if aa_matrix:
                        aa_matrix = np.array(aa_matrix)
                        im = axes[1,1].imshow(aa_matrix.T, cmap='viridis', aspect='auto')
                        axes[1,1].set_xlabel('Peptides')
                        axes[1,1].set_ylabel('Amino Acids')
                        axes[1,1].set_title('AA Composition Heatmap')
                        axes[1,1].set_xticks(range(len(aa_labels)))
                        axes[1,1].set_xticklabels([l.replace('peptide_', 'P') for l in aa_labels], rotation=45)
                        axes[1,1].set_yticks(range(len(sorted_aa[:8])))
                        axes[1,1].set_yticklabels([aa for aa, _ in sorted_aa[:8]])
                        plt.colorbar(im, ax=axes[1,1], label='Fraction')
                
                plt.tight_layout()
                plt.show()

In [8]:
# ## 9. Interaction Analysis and Binding Site Characterization

print(f"\n{'='*60}")
print("BINDING SITE INTERACTION ANALYSIS")
print(f"{'='*60}")

if len(successful_df) > 0:
    # Analysis of binding modes and interactions
    print("Binding Mode Analysis:")
    
    # Group peptides by binding strength
    strong_binders = successful_df[successful_df['Best_Affinity'] < -7.0]['Ligand'].tolist()
    moderate_binders = successful_df[(successful_df['Best_Affinity'] >= -7.0) & 
                                   (successful_df['Best_Affinity'] < -5.5)]['Ligand'].tolist()
    weak_binders = successful_df[successful_df['Best_Affinity'] >= -5.5]['Ligand'].tolist()
    
    print(f"\n🔥 Strong Binders (<-7.0 kcal/mol): {len(strong_binders)}")
    for binder in strong_binders:
        affinity = successful_df[successful_df['Ligand'] == binder]['Best_Affinity'].iloc[0]
        sequence = peptide_sequences.get(binder, 'Unknown')
        print(f"   {binder}: {affinity:.1f} kcal/mol ({sequence})")
    
    print(f"\n⚡ Moderate Binders (-7.0 to -5.5 kcal/mol): {len(moderate_binders)}")
    for binder in moderate_binders:
        affinity = successful_df[successful_df['Ligand'] == binder]['Best_Affinity'].iloc[0]
        sequence = peptide_sequences.get(binder, 'Unknown')
        print(f"   {binder}: {affinity:.1f} kcal/mol ({sequence})")
    
    print(f"\n💧 Weak Binders (>-5.5 kcal/mol): {len(weak_binders)}")
    for binder in weak_binders:
        affinity = successful_df[successful_df['Ligand'] == binder]['Best_Affinity'].iloc[0]
        sequence = peptide_sequences.get(binder, 'Unknown')
        print(f"   {binder}: {affinity:.1f} kcal/mol ({sequence})")
    
    # Binding efficiency analysis
    print(f"\n{'='*40}")
    print("BINDING EFFICIENCY ANALYSIS")
    print(f"{'='*40}")
    
    if 'Length' in successful_df.columns:
        successful_df['Binding_Efficiency'] = -successful_df['Best_Affinity'] / successful_df['Length']
        
        print("Binding Efficiency (|Affinity|/Length):")
        efficiency_ranking = successful_df.nlargest(5, 'Binding_Efficiency')
        
        for idx, row in efficiency_ranking.iterrows():
            print(f"   {row['Ligand']}: {row['Binding_Efficiency']:.3f} (kcal/mol)/AA")
            print(f"      Affinity: {row['Best_Affinity']:.1f} kcal/mol, Length: {row['Length']} AA")



BINDING SITE INTERACTION ANALYSIS


In [9]:
# ## 10. Export Results and Generate Reports

print(f"\n{'='*60}")
print("RESULTS EXPORT AND REPORTING")
print(f"{'='*60}")

# Save detailed results
results_file = analysis_dir / "docking_results_detailed.csv"
df_results.to_csv(results_file, index=False)
print(f"✓ Detailed results saved: {results_file}")

# Save successful dockings with additional analysis
if len(successful_df) > 0:
    successful_file = analysis_dir / "successful_dockings_analysis.csv"
    successful_df.to_csv(successful_file, index=False)
    print(f"✓ Successful dockings analysis saved: {successful_file}")

# Generate summary report
summary_file = analysis_dir / "docking_summary_report.txt"

with open(summary_file, 'w') as f:
    f.write("MOLECULAR DOCKING SUMMARY REPORT\n")
    f.write("=" * 60 + "\n\n")
    
    f.write(f"Receptor: {RECEPTOR_FILE}\n")
    f.write(f"Number of ligands: {len(docking_results)}\n")
    f.write(f"Successful dockings: {len(successful_dockings)}\n")
    f.write(f"Failed dockings: {len(failed_dockings)}\n\n")
    
    f.write("BINDING SITE CONFIGURATION:\n")
    f.write(f"Center: ({BINDING_SITE['center_x']:.2f}, {BINDING_SITE['center_y']:.2f}, {BINDING_SITE['center_z']:.2f})\n")
    f.write(f"Box size: ({BINDING_SITE['size_x']:.1f}, {BINDING_SITE['size_y']:.1f}, {BINDING_SITE['size_z']:.1f})\n\n")
    
    if len(successful_df) > 0:
        f.write("BINDING AFFINITY STATISTICS:\n")
        f.write(f"Best binding affinity: {successful_df['Best_Affinity'].min():.1f} kcal/mol\n")
        f.write(f"Worst binding affinity: {successful_df['Best_Affinity'].max():.1f} kcal/mol\n")
        f.write(f"Mean binding affinity: {successful_df['Best_Affinity'].mean():.1f} kcal/mol\n")
        f.write(f"Standard deviation: {successful_df['Best_Affinity'].std():.1f} kcal/mol\n\n")
        
        f.write("TOP BINDERS:\n")
        top_3 = successful_df.nsmallest(3, 'Best_Affinity')
        for idx, row in top_3.iterrows():
            sequence = peptide_sequences.get(row['Ligand'], 'Unknown')
            f.write(f"{row['Ligand']}: {row['Best_Affinity']:.1f} kcal/mol ({sequence})\n")
        
        f.write("\nBINDING CLASSIFICATION:\n")
        f.write(f"Strong binders (<-7.0 kcal/mol): {len(strong_binders)}\n")
        f.write(f"Moderate binders (-7.0 to -5.5 kcal/mol): {len(moderate_binders)}\n")
        f.write(f"Weak binders (>-5.5 kcal/mol): {len(weak_binders)}\n")
    
    f.write("\nAUTODOCK VINA PARAMETERS:\n")
    for key, value in VINA_CONFIG.items():
        f.write(f"{key}: {value}\n")

print(f"✓ Summary report saved: {summary_file}")

# Create file structure summary
structure_file = analysis_dir / "output_file_structure.txt"

with open(structure_file, 'w') as f:
    f.write("DOCKING OUTPUT FILE STRUCTURE\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("DockingPrep/\n")
    f.write("├── ligands/\n")
    for mol2_file in sorted(ligands_dir.glob("*.mol2")):
        f.write(f"│   ├── {mol2_file.name}\n")
    for pdbqt_file in sorted(ligands_dir.glob("*.pdbqt")):
        f.write(f"│   ├── {pdbqt_file.name}\n")
    
    f.write("├── receptor/\n")
    f.write(f"│   └── {RECEPTOR_FILE}\n")
    
    f.write("├── docking_results/\n")
    for result_file in sorted(results_dir.glob("*")):
        f.write(f"│   ├── {result_file.name}\n")
    
    f.write("└── analysis/\n")
    for analysis_file in sorted(analysis_dir.glob("*")):
        f.write(f"    ├── {analysis_file.name}\n")

print(f"✓ File structure summary saved: {structure_file}")


RESULTS EXPORT AND REPORTING
✓ Detailed results saved: DockingPrep\analysis\docking_results_detailed.csv
✓ Summary report saved: DockingPrep\analysis\docking_summary_report.txt
✓ File structure summary saved: DockingPrep\analysis\output_file_structure.txt


In [10]:
# ## 11. Visualization and Molecular Graphics Recommendations

print(f"\n{'='*60}")
print("VISUALIZATION RECOMMENDATIONS")
print(f"{'='*60}")

visualization_guide = f"""
## MOLECULAR VISUALIZATION GUIDE

### 1. PyMOL Visualization Commands:

```bash
# Load receptor and docked poses
pymol {receptor_dir / RECEPTOR_FILE}

# Load all docked structures
"""

if len(successful_dockings) > 0:
    visualization_guide += f"""
# Example for top binder:
# pymol {results_dir / f"{successful_dockings[0]['ligand']}_docked.pdbqt"}
"""

visualization_guide += f"""
# PyMOL commands for analysis:
hide everything
show cartoon, receptor
show sticks, ligand
color blue, receptor
color red, ligand
center ligand
zoom ligand, 10
```

### 2. ChimeraX Visualization:

```bash
# Open ChimeraX and run:
open {receptor_dir / RECEPTOR_FILE}
"""

if len(successful_dockings) > 0:
    visualization_guide += f"""
# open {results_dir / f"{successful_dockings[0]['ligand']}_docked.pdbqt"}
"""

visualization_guide += f"""
style receptor cartoon
style ligand stick
color receptor blue
color ligand red
view
```

### 3. VMD Visualization:

```tcl
# VMD TCL commands:
mol new {receptor_dir / RECEPTOR_FILE}
"""

if len(successful_dockings) > 0:
    visualization_guide += f"""
# mol new {results_dir / f"{successful_dockings[0]['ligand']}_docked.pdbqt"}
"""

visualization_guide += f"""
mol modstyle 0 0 NewCartoon
mol modcolor 0 0 Blue
mol modstyle 0 1 Licorice
mol modcolor 0 1 Red
```

### 4. Interaction Analysis Tools:

- **PLIP (Protein-Ligand Interaction Profiler)**:
  - Web: https://plip-tool.biotec.tu-dresden.de/plip-web/plip/index
  - Command line: plip -f docked_complex.pdb

- **ProLIF (Protein-Ligand Interaction Fingerprints)**:
  ```python
  import prolif as plf
  # Analyze interactions between receptor and ligands
  ```

- **OpenEye OMEGA** (for conformational analysis):
  ```bash
  omega2 -in ligand.mol2 -out conformers.oeb -maxconfs 100
  ```

### 5. Binding Site Analysis:

- **CASTp** (Computed Atlas of Surface Topography of proteins):
  - Web: http://sts.bioe.uic.edu/castp/
  
- **fpocket** (pocket detection):
  ```bash
  fpocket -f {RECEPTOR_FILE}
  ```

### 6. Quality Assessment:

- **Ramachandran plots** for receptor quality
- **RMSD analysis** of docked poses
- **Binding site volume** calculations
- **Solvent accessible surface area** analysis

### 7. Export for Publication:

```bash
# High-quality figure generation in PyMOL:
ray 1200, 1200
png figure_1.png, dpi=300

# ChimeraX publication figures:
save figure_1.png width 1200 height 1200 supersample 3
```
"""

print(visualization_guide)

# Save visualization guide
viz_file = analysis_dir / "visualization_guide.txt"
with open(viz_file, 'w') as f:
    f.write(visualization_guide)

print(f"✓ Visualization guide saved: {viz_file}")

# ## 12. Final Summary and Next Steps

print(f"\n{'='*60}")
print("FINAL SUMMARY AND NEXT STEPS")
print(f"{'='*60}")

final_summary = f"""
## MOLECULAR DOCKING COMPLETED SUCCESSFULLY! 🎉

### Results Overview:
- Total peptides processed: {len(docking_results)}
- Successful dockings: {len(successful_dockings)}
- Failed dockings: {len(failed_dockings)}
"""

if len(successful_df) > 0:
    best_peptide = successful_df.loc[successful_df['Best_Affinity'].idxmin()]
    final_summary += f"""
- Best binding affinity: {successful_df['Best_Affinity'].min():.1f} kcal/mol ({best_peptide['Ligand']})
- Average binding affinity: {successful_df['Best_Affinity'].mean():.1f} ± {successful_df['Best_Affinity'].std():.1f} kcal/mol
"""

final_summary += f"""
### Files Generated:
📁 Docking Results: {len(list(results_dir.glob('*')))} files in {results_dir}
📁 Analysis Reports: {len(list(analysis_dir.glob('*')))} files in {analysis_dir}

### Key Output Files:
✓ {results_file.name} - Detailed docking results
✓ {summary_file.name} - Summary report
✓ {viz_file.name} - Visualization guide
"""

if len(successful_df) > 0:
    final_summary += f"✓ {successful_file.name} - Successful dockings analysis\n"

final_summary += f"""
### Recommended Next Steps:

1. **Structural Validation**:
   - Visually inspect top binding poses using PyMOL/ChimeraX
   - Verify binding interactions make chemical sense
   - Check for unreasonable conformations

2. **Interaction Analysis**:
   - Use PLIP or ProLIF to analyze protein-ligand interactions
   - Identify key residues involved in binding
   - Map hydrogen bonds, hydrophobic contacts, electrostatic interactions

3. **Binding Site Characterization**:
   - Analyze binding pocket properties (volume, hydrophobicity, charge)
   - Compare with known VEGF inhibitors if available
   - Identify druggable hotspots

4. **Lead Optimization** (if applicable):
   - Focus on peptides with binding affinity < -7.0 kcal/mol
   - Consider structure-based modifications
   - Design analogs with improved properties

5. **Validation Studies**:
   - Consider molecular dynamics simulations for top hits
   - Plan experimental validation (binding assays, cell-based assays)
   - Evaluate selectivity against related targets

6. **Advanced Analysis**:
   - Free energy perturbation (FEP) calculations
   - Pharmacophore modeling
   - ADMET property prediction

### Troubleshooting:
- If docking failed: Check ligand preparation and receptor file format
- Poor binding scores: Consider alternative binding sites or conformations
- Unrealistic poses: Adjust search space or increase exhaustiveness

### Publication/Reporting:
- Include binding site definition and search parameters
- Report statistical significance of results
- Provide validation studies for top hits
- Compare with literature benchmarks if available

## CONGRATULATIONS! Your molecular docking study is complete! 🔬✨
"""

print(final_summary)

# Save final summary
final_summary_file = analysis_dir / "final_summary_and_next_steps.txt"
with open(final_summary_file, 'w') as f:
    f.write(final_summary)

print(f"\n✓ Final summary saved: {final_summary_file}")

print(f"\n{'='*60}")
print("ALL DOCKING AND ANALYSIS TASKS COMPLETED!")
print(f"Check your results in: {base_dir.absolute()}")
print(f"{'='*60}")


VISUALIZATION RECOMMENDATIONS

## MOLECULAR VISUALIZATION GUIDE

### 1. PyMOL Visualization Commands:

```bash
# Load receptor and docked poses
pymol DockingPrep\receptor\1VPF_receptor.pdbqt

# Load all docked structures

# Example for top binder:
# pymol DockingPrep\docking_results\peptide_1_docked.pdbqt

# PyMOL commands for analysis:
hide everything
show cartoon, receptor
show sticks, ligand
color blue, receptor
color red, ligand
center ligand
zoom ligand, 10
```

### 2. ChimeraX Visualization:

```bash
# Open ChimeraX and run:
open DockingPrep\receptor\1VPF_receptor.pdbqt

# open DockingPrep\docking_results\peptide_1_docked.pdbqt

style receptor cartoon
style ligand stick
color receptor blue
color ligand red
view
```

### 3. VMD Visualization:

```tcl
# VMD TCL commands:
mol new DockingPrep\receptor\1VPF_receptor.pdbqt

# mol new DockingPrep\docking_results\peptide_1_docked.pdbqt

mol modstyle 0 0 NewCartoon
mol modcolor 0 0 Blue
mol modstyle 0 1 Licorice
mol modcolor 0 1 Red
```
